# Goodgorithm — Sentiment CNN training

Trains the small CNN that replaces the VADER placeholder in
`processing/src/sentiment.py`. See `CLAUDE.md` in the repo root for full
project context — this is the "ML training" pipeline stage: periodic,
run by a human on a free Colab/Kaggle GPU, not continuous.

**What this notebook does:** loads and harmonizes three public sentiment
datasets into one 3-class (negative/neutral/positive) corpus, trains a
Kim (2014)-style CNN over GloVe-Twitter embeddings, evaluates it, exports
it to ONNX, and uploads it to the `goodgorithm-models` R2 bucket that
`processing/` downloads from at startup.

**Before running:** this notebook fetches
`processing/src/sentiment_model.py` from a specific pinned commit on the
repo's `main` branch (not a live "latest" fetch) so the tokenizer/vocab
logic used here exactly matches what `processing/`'s inference path uses
— they can never drift apart even though they run in totally different
environments. If you've changed that file since, update
`SENTIMENT_MODEL_COMMIT` below to the new commit SHA.


In [ ]:
!pip install -q datasets gensim onnx onnxruntime boto3

import random

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


## Fetch the shared tokenizer/vocab file

Pinned to a commit SHA, not `main` — a later edit to this file shouldn't
silently invalidate a model that's already been published.


In [ ]:
import urllib.request

SENTIMENT_MODEL_COMMIT = "7c6090f0a5b0bf7851421f882a30061a40d9ad25"
SENTIMENT_MODEL_RAW_URL = (
    f"https://raw.githubusercontent.com/goodgorithm/goodgorithm/{SENTIMENT_MODEL_COMMIT}"
    f"/processing/src/sentiment_model.py"
)

urllib.request.urlretrieve(SENTIMENT_MODEL_RAW_URL, "sentiment_model.py")
import sentiment_model

print("EMBEDDING_DIM", sentiment_model.EMBEDDING_DIM)
print("FILTER_SIZES", sentiment_model.FILTER_SIZES)
print("NUM_FILTERS", sentiment_model.NUM_FILTERS)
print("MAX_SEQ_LEN", sentiment_model.MAX_SEQ_LEN)
print("MAX_VOCAB_SIZE", sentiment_model.MAX_VOCAB_SIZE)
print(sentiment_model.tokenize("Check this out https://example.com/x @friend #blessed :) <3"))


## Load + harmonize three datasets

All three get mapped to the same 3-class scheme: `0=negative, 1=neutral,
2=positive` (matches TweetEval's native convention, and is the label
order `processing/src/sentiment.py` assumes when it computes
`P(positive) - P(negative)` from the model's output).

### Sentiment140

~1.6M tweets, weak-labeled via emoticons (which are stripped from the
text itself). Historically the released train split is binary-only
(labels 0/4) even though the label field's nominal range is 0/2/4 — the
code below doesn't assume this, it just maps whatever values are actually
present.


In [ ]:
from datasets import load_dataset

# stanfordnlp/sentiment140's canonical repo uses a legacy loading script,
# which current `datasets` versions no longer execute ("Dataset scripts
# are no longer supported"). The auto-converted parquet revision works
# without a script and has identical content -- verified directly against
# the real dataset before this notebook was written.
sentiment140 = load_dataset("stanfordnlp/sentiment140", revision="refs/convert/parquet")
print(sentiment140)

SENTIMENT140_LABEL_MAP = {0: 0, 2: 1, 4: 2}  # -> negative, neutral, positive

print("label values present in train:", sorted(set(sentiment140["train"]["sentiment"])))

s140_texts = list(sentiment140["train"]["text"])
s140_labels = [SENTIMENT140_LABEL_MAP[v] for v in sentiment140["train"]["sentiment"]]
print(f"sentiment140: {len(s140_texts)} examples")


### TweetEval / SemEval-2017 Task 4 (sentiment config)

Human-annotated, natively 3-class, best domain + label-quality match to
our own short social-post text. `train` + `validation` go into the
combined training pool; the dataset's own `test` split is held out
entirely and used later as a second, domain-matched evaluation number —
not mixed into our train/val/test split, to avoid any leakage.


In [ ]:
tweet_eval = load_dataset("cardiffnlp/tweet_eval", "sentiment")
print(tweet_eval)
# label convention: 0=negative, 1=neutral, 2=positive — matches our target scheme directly

te_train_texts = list(tweet_eval["train"]["text"]) + list(tweet_eval["validation"]["text"])
te_train_labels = list(tweet_eval["train"]["label"]) + list(tweet_eval["validation"]["label"])
te_test_texts = list(tweet_eval["test"]["text"])
te_test_labels = list(tweet_eval["test"]["label"])
print(f"tweet_eval train+val pool: {len(te_train_texts)}, reserved test: {len(te_test_texts)}")


### GoEmotions

58k Reddit comments, 27 fine-grained emotion labels (multi-label) + a
`neutral` class. We only keep single-label examples (multi-label ones are
genuinely ambiguous about overall valence, better excluded than guessed
at), then map through the **official Demszky et al. (2020) sentiment
grouping** — fetched here from the primary source
(`google-research/google-research`), not hand-typed from memory, since
getting this mapping wrong would quietly mislabel a third of the training
data. The paper's own grouping has 12 positive / 11 negative / 4
"ambiguous" categories + neutral; we drop the ambiguous-grouped examples
rather than force them into positive/negative/neutral.


In [ ]:
import urllib.request
import json as jsonlib

SENTIMENT_MAPPING_URL = (
    "https://raw.githubusercontent.com/google-research/google-research/master"
    "/goemotions/data/sentiment_mapping.json"
)
with urllib.request.urlopen(SENTIMENT_MAPPING_URL) as f:
    ge_sentiment_groups = jsonlib.load(f)

print(ge_sentiment_groups)
assert len(ge_sentiment_groups["positive"]) == 12
assert len(ge_sentiment_groups["negative"]) == 11
assert len(ge_sentiment_groups["ambiguous"]) == 4

GE_LABEL_TO_CLASS = {}
for emotion in ge_sentiment_groups["positive"]:
    GE_LABEL_TO_CLASS[emotion] = 2
for emotion in ge_sentiment_groups["negative"]:
    GE_LABEL_TO_CLASS[emotion] = 0
GE_LABEL_TO_CLASS["neutral"] = 1
# "ambiguous"-grouped emotions and multi-label examples are dropped below, not mapped.


In [ ]:
go_emotions = load_dataset("google-research-datasets/go_emotions", "simplified")
print(go_emotions)

ge_names = go_emotions["train"].features["labels"].feature.names


def harmonize_go_emotions(split):
    texts, labels = [], []
    for example in split:
        if len(example["labels"]) != 1:
            continue  # drop multi-label examples — ambiguous overall valence
        emotion = ge_names[example["labels"][0]]
        if emotion not in GE_LABEL_TO_CLASS:
            continue  # dropped "ambiguous"-grouped emotion
        texts.append(example["text"])
        labels.append(GE_LABEL_TO_CLASS[emotion])
    return texts, labels


ge_texts, ge_labels = harmonize_go_emotions(go_emotions["train"])
ge_val_texts, ge_val_labels = harmonize_go_emotions(go_emotions["validation"])
ge_texts += ge_val_texts
ge_labels += ge_val_labels
print(f"go_emotions (single-label, non-ambiguous): {len(ge_texts)} examples")


## Combine + balance

Sentiment140 (1.6M) would otherwise drown out TweetEval and GoEmotions
combined (roughly 100k) — the model would just learn Sentiment140's
weak-labeled, binary-only signal and lose both the neutral class and the
better-labeled/domain-matched data's influence. We cap Sentiment140 at a
bounded multiple of the other two combined, stratified evenly across its
two classes, and additionally correct for whatever imbalance remains via
class-weighted loss.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

SENTIMENT140_MAX_MULTIPLE = 4

other_pool_size = len(te_train_texts) + len(ge_texts)
s140_cap = min(len(s140_texts), SENTIMENT140_MAX_MULTIPLE * other_pool_size)

s140_idx_by_class = {0: [], 1: [], 2: []}
for i, label in enumerate(s140_labels):
    s140_idx_by_class[label].append(i)

# doesn't assume which classes actually have data -- in practice
# sentiment140's train split is binary-only (no neutral), but this stays
# correct even if that turns out not to hold.
present_classes = [c for c, idxs in s140_idx_by_class.items() if idxs]
per_class_cap = s140_cap // len(present_classes)

rng = random.Random(SEED)
s140_sample_idx = []
for label in present_classes:
    idxs = s140_idx_by_class[label]
    rng.shuffle(idxs)
    s140_sample_idx.extend(idxs[:per_class_cap])

s140_texts_sampled = [s140_texts[i] for i in s140_sample_idx]
s140_labels_sampled = [s140_labels[i] for i in s140_sample_idx]

all_texts = s140_texts_sampled + te_train_texts + ge_texts
all_labels = s140_labels_sampled + te_train_labels + ge_labels

print(f"combined pool: {len(all_texts)} examples")
print(
    "class counts (0=neg,1=neu,2=pos):",
    {c: all_labels.count(c) for c in (0, 1, 2)},
)

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    all_texts, all_labels, test_size=0.2, random_state=SEED, stratify=all_labels
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=SEED, stratify=temp_labels
)
print(f"train={len(train_texts)} val={len(val_texts)} test={len(test_texts)}")

class_weights = compute_class_weight(
    class_weight="balanced", classes=np.array([0, 1, 2]), y=np.array(train_labels)
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print("class weights:", class_weights)


## Tokenize + build vocabulary

Uses `sentiment_model.tokenize()` — the exact same function
`processing/`'s inference path uses. Vocab is capped at `MAX_VOCAB_SIZE`
by frequency; `<pad>`/`<unk>` are reserved ids 0/1.


In [ ]:
from collections import Counter

train_tokens = [sentiment_model.tokenize(t) for t in train_texts]

token_counts = Counter()
for tokens in train_tokens:
    token_counts.update(tokens)

MIN_FREQ = 2
vocab_words = [w for w, c in token_counts.most_common() if c >= MIN_FREQ]
vocab_words = vocab_words[: sentiment_model.MAX_VOCAB_SIZE - 2]  # room for pad/unk

vocab = {sentiment_model.PAD_TOKEN: 0, sentiment_model.UNK_TOKEN: 1}
for i, w in enumerate(vocab_words, start=2):
    vocab[w] = i

print(f"vocab size: {len(vocab)}")

lengths = [len(t) for t in train_tokens]
p50, p90, p95, p99 = np.percentile(lengths, [50, 90, 95, 99])
print(f"token length percentiles: p50={p50:.0f} p90={p90:.0f} p95={p95:.0f} p99={p99:.0f}")
print(f"MAX_SEQ_LEN={sentiment_model.MAX_SEQ_LEN} -- if p95 exceeds this, consider raising it")


## Embeddings

`glove-twitter-100` — pretrained on tweets specifically, a strong domain
match for our short, informal social-post text. Rows are fine-tuned
during training (not frozen), which generally outperforms a frozen
embedding for this kind of small classifier.


In [ ]:
import gensim.downloader as gensim_api

glove = gensim_api.load("glove-twitter-100")

embedding_matrix = np.random.normal(0, 0.1, size=(len(vocab), sentiment_model.EMBEDDING_DIM)).astype(
    np.float32
)
embedding_matrix[vocab[sentiment_model.PAD_TOKEN]] = np.zeros(sentiment_model.EMBEDDING_DIM, dtype=np.float32)

found = 0
for word, idx in vocab.items():
    if word in glove:
        embedding_matrix[idx] = glove[word]
        found += 1

print(f"GloVe coverage: {found}/{len(vocab)} ({found / len(vocab):.1%})")


## Model

Kim (2014)-style CNN: parallel 1D convolutions over the embedded sequence
at a few different window widths, global max-pool each, concatenate,
dropout, linear classifier. Built from `sentiment_model`'s constants so
the recorded config and the actual model can never disagree.

Hyperparameters live in `sentiment_model.py` (the shared file), not
hardcoded twice here.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class SentimentCNN(nn.Module):
    def __init__(self, embedding_matrix, filter_sizes, num_filters, dropout):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix), freeze=False, padding_idx=0
        )
        self.convs = nn.ModuleList(
            [nn.Conv1d(embedding_dim, num_filters, kernel_size=k) for k in filter_sizes]
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), 3)

    def forward(self, input_ids):
        x = self.embedding(input_ids)          # (batch, seq_len, emb_dim)
        x = x.transpose(1, 2)                    # (batch, emb_dim, seq_len)
        pooled = []
        for conv in self.convs:
            c = F.relu(conv(x))                   # (batch, num_filters, L')
            pooled.append(F.max_pool1d(c, c.shape[2]).squeeze(2))
        x = torch.cat(pooled, dim=1)                # (batch, num_filters * len(filter_sizes))
        x = self.dropout(x)
        return self.fc(x)                             # (batch, 3) logits


model = SentimentCNN(
    embedding_matrix,
    sentiment_model.FILTER_SIZES,
    sentiment_model.NUM_FILTERS,
    sentiment_model.DROPOUT,
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
print(f"total params: {num_params:,}")


## Train

Adam, early stopping on validation macro-F1 (patience=2).


In [ ]:
from torch.utils.data import DataLoader, Dataset


class SentimentDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = sentiment_model.encode(sentiment_model.tokenize(self.texts[idx]), self.vocab)
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)


BATCH_SIZE = 256
train_loader = DataLoader(
    SentimentDataset(train_texts, train_labels, vocab), batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(SentimentDataset(val_texts, val_labels, vocab), batch_size=BATCH_SIZE)
test_loader = DataLoader(SentimentDataset(test_texts, test_labels, vocab), batch_size=BATCH_SIZE)


In [ ]:
from sklearn.metrics import f1_score

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

MAX_EPOCHS = 10
PATIENCE = 2

best_val_f1 = -1.0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for batch_ids, batch_labels in train_loader:
        batch_ids, batch_labels = batch_ids.to(DEVICE), batch_labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(batch_ids)
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_ids.size(0)

    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for batch_ids, batch_labels in val_loader:
            batch_ids = batch_ids.to(DEVICE)
            logits = model(batch_ids)
            val_preds.extend(logits.argmax(dim=1).cpu().tolist())
            val_true.extend(batch_labels.tolist())
    val_f1 = f1_score(val_true, val_preds, average="macro")

    print(f"epoch {epoch}: train_loss={total_loss / len(train_texts):.4f} val_macro_f1={val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"early stopping (no val improvement for {PATIENCE} epochs)")
            break

model.load_state_dict(best_state)
print(f"best val macro-F1: {best_val_f1:.4f}")


## Evaluate

Held-out test split from our own combined pool, *plus* TweetEval's
reserved native test split (best domain-match + human-labeled dataset,
so its own official benchmark split is a meaningful independent check
that was never trained on).


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix


def evaluate(texts, labels, vocab, model, label_name):
    loader = DataLoader(SentimentDataset(texts, labels, vocab), batch_size=BATCH_SIZE)
    model.eval()
    preds, true = [], []
    with torch.no_grad():
        for batch_ids, batch_labels in loader:
            batch_ids = batch_ids.to(DEVICE)
            logits = model(batch_ids)
            preds.extend(logits.argmax(dim=1).cpu().tolist())
            true.extend(batch_labels.tolist())

    print(f"--- {label_name} ---")
    print(classification_report(true, preds, target_names=["negative", "neutral", "positive"]))
    print("confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(true, preds))
    return preds, true


_ = evaluate(test_texts, test_labels, vocab, model, "held-out test split (combined pool)")
_ = evaluate(te_test_texts, te_test_labels, vocab, model, "TweetEval's own reserved test split")


In [ ]:
# Spot-check against a handful of examples. Replace these with real
# examples copied from Supabase `staging.raw_posts` for a domain-specific
# eyeball check -- this notebook has no DB access of its own.
spot_check_examples = [
    "Just adopted the sweetest rescue dog, he already knows how to sit!",
    "Traffic was awful this morning and I spilled coffee on my shirt.",
    "The city council meeting is scheduled for 3pm on Tuesday.",
    "Wow. Just wow. That is the kind of exceptionalist twaddle I'd expect.",
]

model.eval()
with torch.no_grad():
    for text in spot_check_examples:
        ids = torch.tensor(
            [sentiment_model.encode(sentiment_model.tokenize(text), vocab)], dtype=torch.long
        ).to(DEVICE)
        probs = F.softmax(model(ids), dim=1)[0].cpu().numpy()
        score = float(probs[2] - probs[0])
        print(f"{score:+.3f}  neg={probs[0]:.2f} neu={probs[1]:.2f} pos={probs[2]:.2f}  :: {text!r}")


## Export to ONNX

`processing/`'s inference path only needs `onnxruntime`, not full
PyTorch — the exported graph bakes in the final softmax so
`sentiment.py` never reimplements it, and the graph is self-contained
(no separately-loaded architecture code needed at inference time, which
is what makes the ONNX approach immune to architecture drift). Fixed
batch=1 / seq_len=`MAX_SEQ_LEN` matches how `score_sentiment()` actually
calls it — one post at a time, no batching.


In [ ]:
class InferenceWrapper(nn.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.model = trained_model

    def forward(self, input_ids):
        logits = self.model(input_ids)
        return F.softmax(logits, dim=1)


export_model = InferenceWrapper(model).to(DEVICE).eval()
dummy_input = torch.zeros((1, sentiment_model.MAX_SEQ_LEN), dtype=torch.long, device=DEVICE)

torch.onnx.export(
    export_model,
    dummy_input,
    "model.onnx",
    input_names=["input_ids"],
    output_names=["probs"],
    opset_version=17,
    dynamo=False,  # forces the stable TorchScript-based exporter; the
    # newer dynamo-based one (PyTorch's new default on some versions)
    # needs the separate `onnxscript` package, which we don't install
)
print("exported model.onnx")


In [ ]:
import onnxruntime as ort

session = ort.InferenceSession("model.onnx", providers=["CPUExecutionProvider"])

with torch.no_grad():
    for text in spot_check_examples:
        ids = sentiment_model.encode(sentiment_model.tokenize(text), vocab)
        ids_np = np.array([ids], dtype=np.int64)

        torch_probs = export_model(torch.tensor(ids_np, device=DEVICE)).cpu().numpy()[0]
        onnx_probs = session.run(None, {"input_ids": ids_np})[0][0]

        assert np.allclose(torch_probs, onnx_probs, atol=1e-4), (text, torch_probs, onnx_probs)

print("ONNX output matches PyTorch output on all spot-check examples")


## Package config.json

Audit metadata only — dataset composition, eval numbers, the commit this
was trained against. Not load-bearing for correctness (the ONNX graph is
self-contained and vocab.json is the only other file inference actually
needs).


In [ ]:
import datetime
import json as jsonlib

VERSION = "v1"  # bump manually for each new published model

model_config = {
    "version": VERSION,
    "architecture_source_commit": SENTIMENT_MODEL_COMMIT,
    "embedding_dim": sentiment_model.EMBEDDING_DIM,
    "filter_sizes": list(sentiment_model.FILTER_SIZES),
    "num_filters": sentiment_model.NUM_FILTERS,
    "dropout": sentiment_model.DROPOUT,
    "max_seq_len": sentiment_model.MAX_SEQ_LEN,
    "vocab_size": len(vocab),
    "embedding_source": "glove-twitter-100",
    "trained_at": datetime.datetime.utcnow().isoformat() + "Z",
    "dataset_composition": {
        "sentiment140_sampled": len(s140_texts_sampled),
        "tweet_eval_train_val": len(te_train_texts),
        "go_emotions_single_label": len(ge_texts),
        "combined_total": len(all_texts),
    },
    "best_val_macro_f1": best_val_f1,
}

with open("vocab.json", "w") as f:
    jsonlib.dump(vocab, f)
with open("config.json", "w") as f:
    jsonlib.dump(model_config, f, indent=2)

print(jsonlib.dumps(model_config, indent=2))


## Upload to R2

Cloudflare R2 is S3-API-compatible, so `boto3`'s S3 client works directly
against it. Provide credentials via Colab Secrets / Kaggle Secrets
(preferred) or paste them into the fallback cell below — either way,
**never commit real credentials into this notebook's output**.

Publishing to a specific version directory always happens; **flipping
`latest.json` (what `processing/` actually reads by default) is a
separate, explicitly-gated step** — so a blind "Run All" publishes the
versioned artifacts without silently promoting them to production.


In [ ]:
R2_ACCOUNT_ID = R2_ACCESS_KEY_ID = R2_SECRET_ACCESS_KEY = R2_BUCKET_NAME = None

try:
    from google.colab import userdata

    R2_ACCOUNT_ID = userdata.get("R2_ACCOUNT_ID")
    R2_ACCESS_KEY_ID = userdata.get("R2_ACCESS_KEY_ID")
    R2_SECRET_ACCESS_KEY = userdata.get("R2_SECRET_ACCESS_KEY")
    R2_BUCKET_NAME = userdata.get("R2_BUCKET_NAME")
except Exception as e:
    # covers both "not running in Colab" (ImportError) and Colab Secrets
    # being unreachable in this execution context -- e.g. a non-interactive
    # / background run raises TimeoutException instead, not ImportError,
    # since the one-time permission prompt needs a live UI session
    print(f"Colab secrets unavailable ({type(e).__name__}: {e}), trying Kaggle secrets...")

if not R2_ACCOUNT_ID:
    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
        R2_ACCOUNT_ID = secrets.get_secret("R2_ACCOUNT_ID")
        R2_ACCESS_KEY_ID = secrets.get_secret("R2_ACCESS_KEY_ID")
        R2_SECRET_ACCESS_KEY = secrets.get_secret("R2_SECRET_ACCESS_KEY")
        R2_BUCKET_NAME = secrets.get_secret("R2_BUCKET_NAME")
    except Exception as e:
        print(f"Kaggle secrets unavailable ({type(e).__name__}: {e}), falling back to manual values...")

if not R2_ACCOUNT_ID:
    # Manual fallback -- fill these in locally, never commit real values.
    R2_ACCOUNT_ID = ""
    R2_ACCESS_KEY_ID = ""
    R2_SECRET_ACCESS_KEY = ""
    R2_BUCKET_NAME = ""

assert R2_ACCOUNT_ID and R2_ACCESS_KEY_ID and R2_SECRET_ACCESS_KEY and R2_BUCKET_NAME, (
    "R2 credentials not set -- see the markdown cell above"
)


In [ ]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto",
)

PREFIX = f"sentiment-cnn/{VERSION}"
s3.upload_file("model.onnx", R2_BUCKET_NAME, f"{PREFIX}/model.onnx")
s3.upload_file("vocab.json", R2_BUCKET_NAME, f"{PREFIX}/vocab.json")
s3.upload_file("config.json", R2_BUCKET_NAME, f"{PREFIX}/config.json")
print(f"uploaded to s3://{R2_BUCKET_NAME}/{PREFIX}/")


In [ ]:
# Deliberately gated -- flip to True only after checking the eval numbers
# and spot-check output above. This is what actually makes processing/
# start using this model (it reads sentiment-cnn/latest.json by default).
#
# NOTE (2026-08-12): this only flips latest.json. goodgorithm-models is a
# private R2 bucket, and Colab has no `gh`/repo access, so this cell alone
# cannot make the version publicly downloadable -- it does NOT fulfill the
# "we open-source model weights" commitment on its own. Prefer running
# `cd training && uv run python r2_release.py publish <version>` instead
# (from a machine with an authenticated `gh` CLI) -- it does both the
# latest.json flip and the public GitHub Release in one step. If you do
# use this cell, still run r2_release.py publish afterward; it's
# idempotent on the latest.json flip and will just create the missing
# release. See the release-sentiment-model skill.
PUBLISH_AS_LATEST = False

if PUBLISH_AS_LATEST:
    import io
    import json as jsonlib

    latest_bytes = jsonlib.dumps({"version": VERSION}).encode()
    s3.upload_fileobj(io.BytesIO(latest_bytes), R2_BUCKET_NAME, "sentiment-cnn/latest.json")
    print(f"sentiment-cnn/latest.json now points to {VERSION}")
    print("Reminder: run `r2_release.py publish` too, to create the public GitHub Release.")
else:
    print("PUBLISH_AS_LATEST is False -- latest.json untouched, this version is published but not live")
